In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
import joblib

In [5]:
# =========================
# 1) ตั้งค่า Path Dataset
# =========================
# ปรับ path 
data_dir = "Garbage classification"

img_size = (224, 224)
batch_size = 32
num_classes = 6
random_state = 42


In [6]:
# =========================
# 2) โหลดรูป (ไม่ทำ augment ตอนดึง feature)
# =========================
datagen = ImageDataGenerator(rescale=1./255)

data = datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False  # สำคัญ: เพื่อให้ labels ตรงกับ features
)

class_indices = data.class_indices
print("Class mapping:", class_indices)

Found 2527 images belonging to 6 classes.
Class mapping: {'cardboard': 0, 'glass': 1, 'metal': 2, 'paper': 3, 'plastic': 4, 'trash': 5}


In [7]:
# =========================
# 3) โหลด Pretrained CNN
# =========================
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    pooling='avg',              # ได้ feature vector โดยตรง
    input_shape=(224, 224, 3)
)


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [11]:
# =========================
# 4) ดึง Feature ทั้งชุด
# =========================
features = base_model.predict(data, verbose=1)
labels = data.classes  # เป็นเลข 0-5 ตามโฟลเดอร์

print("Feature shape:", features.shape)
print("Labels shape:", labels.shape)

79/79 ━━━━━━━━━━━━━━━━━━━━ 31s 393ms/step
Feature shape: (2527, 1280)
Labels shape: (2527,)


In [12]:
# =========================
# 5) แบ่ง Train/Test
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    features, labels,
    test_size=0.2,
    random_state=random_state,
    stratify=labels
)


In [19]:
# =========================
# 6) สร้าง 3 โมเดล ML
# =========================
rf = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

svm = SVC(
    kernel='rbf',
    probability=True,
    random_state=random_state
)

xgb = XGBClassifier(
    n_estimators=300,
    eval_metric='mlogloss',
    scale_pos_weight=1
)

In [20]:
# =========================
# 7) รวมเป็น Ensemble (Soft Voting)
# =========================
ensemble = VotingClassifier(
    estimators=[
        ('rf', rf),
        ('svm', svm),
        ('xgb', xgb)
    ],
    voting='soft'   # ใช้ probability โหวต
)

In [21]:
# =========================
# 8) เทรน
# =========================
ensemble.fit(X_train, y_train)


c:\Users\podja\AppData\Local\Programs\Python\Python310\lib\site-packages\xgboost\training.py:200: UserWarning: [23:11:48] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "scale_pos_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,estimators,"[('rf', ...), ('svm', ...), ...]"
,voting,'soft'
,weights,None
,n_jobs,None
,flatten_transform,True
,verbose,False
,n_estimators,300
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1


In [22]:
# =========================
# 9) ประเมินผล
# =========================
y_pred = ensemble.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print("\nEnsemble Accuracy:", acc)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))


Ensemble Accuracy: 0.8181818181818182

Classification Report:

              precision    recall  f1-score   support

           0       0.88      0.85      0.87        81
           1       0.79      0.87      0.83       100
           2       0.81      0.79      0.80        82
           3       0.80      0.89      0.84       119
           4       0.82      0.77      0.80        97
           5       0.80      0.44      0.57        27

    accuracy                           0.82       506
   macro avg       0.82      0.77      0.79       506
weighted avg       0.82      0.82      0.81       506


Confusion Matrix:

[[ 69   0   0   9   2   1]
 [  0  87   8   1   4   0]
 [  0   8  65   3   6   0]
 [  8   1   2 106   1   1]
 [  0  12   3   6  75   1]
 [  1   2   2   7   3  12]]


In [ ]:
# =========================
# 10) เซฟโมเดล
# =========================
joblib.dump(ensemble, "garbage_ensemble_model.pkl")

print("\nSaved: garbage_ensemble_model.pkl ✅")


Saved: garbage_ensemble_model.pkl ✅
